# Sun et al. continuous-age and extreme-age expression comparisons

This notebook contains two related analyses:

1. **Continuous-age comparison (Figure 4C):**
   Compare continuous-age ALDEx effect estimates with the published Sun et al. rank-based trajectory statistics by asking how strongly each model's effects correlate with:
   - changes in raw mean expression; and
   - changes in the fraction of cells expressing each gene.

2. **Extreme-age comparison (Supplementary Figure S2D):**
   Restrict the Sun et al. dataset to the youngest and oldest age groups, which were paired within batches, and compare total transcript and detected-gene distributions without introducing the intermediate age groups.

The calculations and thresholds follow the original analtsis.

## 1. Imports and configuration

In [ ]:
from __future__ import annotations

from pathlib import Path

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import scanpy as sc
from scipy.stats import pearsonr, spearmanr

from scale_aware_st import (
    RepositoryConfig,
    index_result_tables,
    load_pooled_aldex_spec,
)


# ---------------------------------------------------------------------
# Project paths
# ---------------------------------------------------------------------
config = RepositoryConfig.from_env()
ALDEX_RESULT_MODE = "published"  # choose "published" or "recompute"
PROJECT_DIR = config.root
DEG_DIR = config.results_dir / "DEG"
FIGURE_DIR = config.results_dir / "figures"

SUN_H5AD = config.external_data_dir / "sun_et_al" / "aging_coronal.h5ad"
SUN_CONTINUOUS_ALDEX = DEG_DIR / "aldex_cs_ct_all_sig_cont_results.xlsx"
SUN_PUBLISHED_TRAJECTORIES = (
    config.external_data_dir / "sun_et_al"
    / "2023-12-22736D-TableS7_GeneClassificationTrajectory.xlsx"
)

DEG_DIR.mkdir(parents=True, exist_ok=True)
FIGURE_DIR.mkdir(parents=True, exist_ok=True)


# Youngest and oldest age groups used for the extreme-age comparison.
YOUNGEST_AGES = [3.4, 3.8, 4.3, 5.4]
OLDEST_AGES = [30.9, 32.6, 33.2, 34.5]
EXTREME_AGES = YOUNGEST_AGES + OLDEST_AGES

CORRELATION_COLUMNS = [
    "Raw_Mean_Exp_Spearman_rho",
    "Raw_Mean_Exp_Spearman_p",
    "Raw_Mean_Exp_Pearson_r",
    "Raw_Mean_Exp_Pearson_p",
    "Pct_Expressing_Spearman_rho",
    "Pct_Expressing_Spearman_p",
    "Pct_Expressing_Pearson_r",
    "Pct_Expressing_Pearson_p",
]
from scale_aware_st import (
    RepositoryConfig,
    index_result_tables,
    load_pooled_aldex_spec,
)


## 2. Helper functions

In [ ]:
def read_excel_sheets(
    workbook: Path,
    *,
    index_column: str = "gene",
) -> dict[str, pd.DataFrame]:
    """Read all sheets from an Excel workbook and index each by gene."""
    results: dict[str, pd.DataFrame] = {}

    for sheet_name in pd.ExcelFile(workbook).sheet_names:
        table = pd.read_excel(workbook, sheet_name=sheet_name)

        unnamed = [
            column for column in table.columns
            if str(column).startswith("Unnamed:")
        ]
        if unnamed:
            table = table.drop(columns=unnamed)

        table = table.set_index(index_column)
        table.index.name = None
        results[sheet_name] = table

    return results


def add_raw_expression_summaries(
    results: dict[str, pd.DataFrame],
    adata,
    *,
    result_keys_include_region: bool,
) -> dict[str, pd.DataFrame]:
    """
    Add young/old raw mean expression and percent-expressing summaries.

    The original notebook matched ALDEx sheets to cell type and, where present,
    anatomical region using the sheet name.
    """
    gene_ids = adata.var_names.astype(str)

    for key, table in results.items():
        key_parts = key.split("_")

        if (
            result_keys_include_region
            and "_" in key
            and not key.startswith("all_")
        ):
            region, cell_type = key.split("_", 1)

            mask = (
                adata.obs["celltype"]
                .astype(str)
                .str.lower()
                .str.contains(cell_type, na=False)
                &
                adata.obs["region"]
                .astype(str)
                .str.replace("/", "-", regex=False)
                .str.contains(region, na=False)
            )
        else:
            cell_type = key.removeprefix("all_")
            mask = (
                adata.obs["celltype"]
                .astype(str)
                .str.lower()
                .str.contains(cell_type, na=False)
            )

        subset = adata[mask].copy()

        for condition in ("Yng", "Old"):
            table[f"{condition}_Raw_Mean_Exp"] = np.nan
            table[f"{condition}_Pct_Expressing"] = np.nan

        for gene in table.index:
            gene_mask = gene_ids == str(gene)
            if not gene_mask.any():
                continue

            for condition in ("Yng", "Old"):
                condition_subset = subset[subset.obs["age"] == condition]
                values = condition_subset.layers["raw_counts"][:, gene_mask]

                if values.shape[0] == 0:
                    continue

                table.loc[gene, f"{condition}_Raw_Mean_Exp"] = float(values.mean())
                table.loc[gene, f"{condition}_Pct_Expressing"] = (
                    float((values > 0).sum()) / values.shape[0] * 100
                )

    return results


def calculate_model_correlations(
    results: dict[str, pd.DataFrame],
    *,
    effect_column: str,
) -> pd.DataFrame:
    """Correlate model effects with raw-expression and detection changes."""
    output = pd.DataFrame(
        index=results.keys(),
        columns=CORRELATION_COLUMNS,
        dtype=float,
    )

    for key, table in results.items():
        for summary in ("Raw_Mean_Exp", "Pct_Expressing"):
            table[f"Diff_{summary}"] = (
                table[f"Old_{summary}"] - table[f"Yng_{summary}"]
            )

            paired = table[[effect_column, f"Diff_{summary}"]].dropna()

            if len(paired) < 2:
                continue

            output.loc[key, f"{summary}_Spearman_rho"], \
                output.loc[key, f"{summary}_Spearman_p"] = spearmanr(
                    paired[effect_column],
                    paired[f"Diff_{summary}"],
                )

            output.loc[key, f"{summary}_Pearson_r"], \
                output.loc[key, f"{summary}_Pearson_p"] = pearsonr(
                    paired[effect_column],
                    paired[f"Diff_{summary}"],
                )

    return output


def build_published_sun_results(
    trajectory_table: pd.DataFrame,
    adata,
) -> dict[str, pd.DataFrame]:
    """
    Reconstruct significant published Sun et al. trajectory results by cell type.

    The original thresholds are retained:
    - Spearman > 0.3 with lower 95% CI > 0, or
    - Spearman < -0.3 with upper 95% CI < 0.
    """
    output: dict[str, pd.DataFrame] = {}
    gene_ids = adata.var_names.astype(str)

    for cell_type in trajectory_table.columns.str.split("_").str[0].unique():
        columns = [
            column for column in trajectory_table.columns
            if column.startswith(f"{cell_type}_")
        ]
        cell_table = trajectory_table.loc[:, columns].copy()

        significant = (
            (
                (cell_table[f"{cell_type}_Spearman"] > 0.3)
                & (cell_table[f"{cell_type}_Lower95CI"] > 0)
            )
            |
            (
                (cell_table[f"{cell_type}_Spearman"] < -0.3)
                & (cell_table[f"{cell_type}_Upper95CI"] < 0)
            )
        )

        result = (
            cell_table.loc[significant, [f"{cell_type}_Spearman"]]
            .rename(columns={f"{cell_type}_Spearman": "Spearman"})
        )

        subset = adata[
            adata.obs["celltype"].astype(str).eq(cell_type.replace(" ", "-"))
        ].copy()

        for condition in ("Yng", "Old"):
            result[f"{condition}_Raw_Mean_Exp"] = np.nan
            result[f"{condition}_Pct_Expressing"] = np.nan

        for gene in result.index:
            gene_mask = gene_ids == str(gene)
            if not gene_mask.any():
                continue

            for condition in ("Yng", "Old"):
                condition_subset = subset[subset.obs["age"] == condition]
                values = condition_subset.layers["raw_counts"][:, gene_mask]

                if values.shape[0] == 0:
                    continue

                result.loc[gene, f"{condition}_Raw_Mean_Exp"] = float(values.mean())
                result.loc[gene, f"{condition}_Pct_Expressing"] = (
                    float((values > 0).sum()) / values.shape[0] * 100
                )

        output[cell_type] = result

    return output

## 3. Load the Sun et al. data and pre-/post-process ALDEx

In [ ]:
sun = sc.read_h5ad(SUN_H5AD)

sun.layers["raw_counts"] = sun.X.copy()

sun.obs["celltype"] = (
    sun.obs["celltype"]
    .astype(str)
    .str.replace(" ", "-", regex=False)
)
sun.obs["celltype"] = pd.Categorical(sun.obs["celltype"])

print(sun)

if ALDEX_RESULT_MODE == "recompute":
    import kimlabspatial.differential_expression as de
    
    CONTINUOUS_CT_INPUT_DIR = DEG_DIR / "cross_study_cont"
    CONTINUOUS_CT_INPUT_DIR.mkdir(parents=True, exist_ok=True)
    
    # Full coronal dataset; age remains numeric for the continuous-age model.
    de.prep_for_aldex2(
        sun,
        str(CONTINUOUS_CT_INPUT_DIR),
        "cell_type_all",
        obs_key="celltype",
        count_layer=False,
    )

if ALDEX_RESULT_MODE == "recompute":
    from glob import glob

    continuous_ct_raw_files = sorted(
        glob(str(DEG_DIR / "cross_study_cont_results" / "*.xlsx"))
    )

    de.postprocess_aldex2(
        aldex_results=continuous_ct_raw_files,
        output_dir=DEG_DIR,
        filename_prefix="cross_study_ct_",       # exact raw-file prefix
        filename_suffix="_cont_it_results_05122026",       # exact raw-file suffix
        output_stem="aldex_cs_ct_all_sig_cont_results",
        experimental_column="age_scaled",
        pivot_filename=None,
        significant=True,
        make_pivot=False,
        signed=False,
    )
elif ALDEX_RESULT_MODE == "published":
    continuous_ct_pooled = load_pooled_aldex_spec(config.additional_data_dir, "sun_continuous_celltype")
    print(f"Loaded {len(continuous_ct_pooled)} pooled continuous-age cell-type strata.")
else:
    raise ValueError("ALDEX_RESULT_MODE must be published or recompute")


## 4. Extreme-age subset used for raw-expression summaries

In [ ]:
sun_extreme = sun[sun.obs["age"].isin(EXTREME_AGES)].copy()

sun_extreme.obs["numerical_age"] = sun_extreme.obs["age"].astype(float)
sun_extreme.obs["age"] = np.where(
    sun_extreme.obs["numerical_age"] > 10,
    "Old",
    "Yng",
)
sun_extreme.obs["age"] = pd.Categorical(
    sun_extreme.obs["age"],
    categories=["Yng", "Old"],
    ordered=True,
)

display(
    sun_extreme.obs.groupby(
        ["age", "numerical_age"],
        observed=True,
    ).size().to_frame("n_cells")
)

## 5. Load continuous-age ALDEx results

In [ ]:
aldex_results = (
    index_result_tables(load_pooled_aldex_spec(config.additional_data_dir, "sun_continuous_celltype"))
    if ALDEX_RESULT_MODE == "published"
    else read_excel_sheets(SUN_CONTINUOUS_ALDEX)
)

aldex_results = add_raw_expression_summaries(
    aldex_results,
    sun_extreme,
    result_keys_include_region=True,
)

# The downstream comparison is at the cell-type level. This reproduces the
# original key simplification after all-region results were loaded.
aldex_results_by_cell_type = (
    aldex_results
    if ALDEX_RESULT_MODE == "published"
    else {
        key.removeprefix("all_"): table
        for key, table in aldex_results.items()
        if key.startswith("all_")
    }
)

print(sorted(aldex_results_by_cell_type))

## 6. Load and reconstruct the published Sun et al. trajectory results

In [ ]:
published_table = pd.read_excel(SUN_PUBLISHED_TRAJECTORIES)
published_table = published_table.set_index("Gene")
published_table.index.name = None

# Retain the first 72 trajectory-statistic columns, matching the source notebook.
published_table = published_table.iloc[:, :72]

published_results = build_published_sun_results(
    published_table,
    sun_extreme,
)

published_results = {
    key.lower().replace(" ", "-"): table
    for key, table in published_results.items()
}

## 7. Correlate model effects with raw-expression changes

In [ ]:
published_correlations = calculate_model_correlations(
    published_results,
    effect_column="Spearman",
)

aldex_correlations = calculate_model_correlations(
    aldex_results_by_cell_type,
    effect_column="age_scaled:est",
)

display(published_correlations)
display(aldex_correlations)

## 8. Figure 4C: correlation with raw expression and detection fraction

In [ ]:
raw_df = pd.concat(
    [
        published_correlations["Raw_Mean_Exp_Pearson_r"],
        aldex_correlations["Raw_Mean_Exp_Pearson_r"],
    ],
    axis=1,
).dropna()
raw_df.columns = ["Sun_Method", "ALDEx"]

pct_df = pd.concat(
    [
        published_correlations["Pct_Expressing_Pearson_r"],
        aldex_correlations["Pct_Expressing_Pearson_r"],
    ],
    axis=1,
).dropna()
pct_df.columns = ["Sun_Method", "ALDEx"]

plot_df = raw_df.join(
    pct_df,
    lsuffix="_raw",
    rsuffix="_pct",
).sort_index(ascending=False)

plot_df = plot_df.drop(
    index=["neutrophil", "vlmc", "b-cell"],
    errors="ignore",
)

y_positions = np.arange(len(plot_df))

figure, axes = plt.subplots(
    1,
    2,
    figsize=(8, 6),
    sharey=True,
)

# Raw mean expression
sun_values = plot_df["Sun_Method_raw"].astype(float).values
aldex_values = plot_df["ALDEx_raw"].astype(float).values

for index in range(len(plot_df)):
    axes[0].plot(
        [sun_values[index], aldex_values[index]],
        [index, index],
        linewidth=1.5,
        alpha=0.7,
    )

axes[0].scatter(sun_values, y_positions, s=70, label="Sun method", alpha=0.9)
axes[0].scatter(aldex_values, y_positions, s=70, label="ALDEx", alpha=0.9)
axes[0].set_title("Raw mean expression")
axes[0].set_xlabel("Pearson r")
axes[0].set_yticks(y_positions)
axes[0].set_yticklabels(plot_df.index)
axes[0].set_xlim(0.3, 1)

# Percent expressing
sun_values = plot_df["Sun_Method_pct"].astype(float).values
aldex_values = plot_df["ALDEx_pct"].astype(float).values

for index in range(len(plot_df)):
    axes[1].plot(
        [sun_values[index], aldex_values[index]],
        [index, index],
        linewidth=1.5,
        alpha=0.7,
    )

axes[1].scatter(sun_values, y_positions, s=70, label="Sun method", alpha=0.9)
axes[1].scatter(aldex_values, y_positions, s=70, label="ALDEx", alpha=0.9)
axes[1].set_title("% expressing")
axes[1].set_xlabel("Pearson r")
axes[1].set_xlim(0.75, 1)

axes[1].legend(
    bbox_to_anchor=(1.02, 1),
    loc="upper left",
    frameon=False,
)

figure.suptitle(
    "Cross-study concordance with raw expression and detection rates",
    y=0.98,
)
plt.tight_layout()
plt.savefig(
    FIGURE_DIR / "fig4c_sun_continuous_raw_expression_concordance.pdf",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

## 9. Supplementary Figure S2D: extreme-age QC distributions

In [ ]:
sc.pp.calculate_qc_metrics(
    sun_extreme,
    qc_vars=(),
    percent_top=None,
    log1p=False,
    inplace=True,
)

In [ ]:
figure, axis = plt.subplots(figsize=(6, 9))

sc.pl.violin(
    sun_extreme,
    "total_counts",
    groupby="age",
    stripplot=False,
    inner="box",
    show=False,
    ax=axis,
)

axis.grid(True, axis="y", linestyle="--", alpha=0.5)
axis.set_ylim(0, 2500)

plt.savefig(
    FIGURE_DIR / "supp_fig_s2d_sun_extreme_age_total_counts.pdf",
    dpi=600,
    bbox_inches="tight",
)
plt.show()

In [ ]:
figure, axis = plt.subplots(figsize=(6, 9))

sc.pl.violin(
    sun_extreme,
    "n_genes_by_counts",
    groupby="age",
    stripplot=False,
    inner="box",
    show=False,
    ax=axis,
)

axis.grid(True, axis="y", linestyle="--", alpha=0.5)

plt.savefig(
    FIGURE_DIR / "supp_fig_s2d_sun_extreme_age_detected_genes.pdf",
    dpi=600,
    bbox_inches="tight",
)
plt.show()